# ERK-KTR whole-FOV stimulation on the Ti2-E (df_acquire API)

The same kind of experiment as `stim_rtmsequence.ipynb`, built with
the **legacy `df_acquire` API**: one pandas DataFrame row per FOV and
timestep, converted to events with `df_to_events()`.

**When to use which API:**
- **RTMSequence** (recommended): phases are declared as `RTMSequence` objects and joined with `combine`.
- **df_acquire** (this notebook): full control over the acquisition table, e.g. per-FOV treatments on a well plate or stim schedules that do not fit into phases.

**Workflow:**
1. Connect to the Ti2-E, define channels and stimulation treatments
2. Set up the pipeline
3. Pick FOV positions in napari and build one `df_acquire` per phase
4. Convert to events and run; the second phase continues the first
5. Merge the tracks into `exp_data.parquet`

**Channels:**

| Role | Jungfrau preset | Ti2-E preset |
|------|-----------------|--------------|
| Nuclear marker (H2B-miRFP), segmented | `miRFP` | `iRFP` |
| ERK-KTR reporter (mScarlet3) | `mScarlet3` | `RFP` |
| Stimulation, blue light, whole FOV | `CyanStim` | `GFP` |
| Optocheck (mCitrine) | `mCitrine` | `YFP_Custom` |

The Ti2-E has no DMD, so stimulation lights the whole field of view with the
blue excitation of the ET-GFP cube. The optocheck channel is excited by the
same cube, so it also stimulates; that is why it runs on the last frame only.
All presets share the X-Cite XT600 LED; the `power` of a `PowerChannel` sets
its intensity in percent. Exposures and powers are the Jungfrau values:
tune them for the XT600 before a real run.

### Imports

The df_acquire API uses `Channel`, `StimTreatment` and
`SegmentationMethod`. With RTMSequence, `PowerChannel` plus
`stim_channels`/`stim_frames` replace `StimTreatment`.

In [ ]:
import os
import time

import pandas as pd

from faro.core.data_structures import (
    Channel,  # imaging channel (config, exposure, group)
    StimTreatment,  # df_acquire only: stim schedule per condition
    SegmentationMethod,
)
import faro.core.utils as utils

### Microscope

`Ti2E()` loads `NikonTi2e.cfg`, applies `System/Startup` and selects the
`Channel` group, so channels need no `group`. Close NIS-Elements first.

In [ ]:
from faro_aoki.ti2e import Ti2E

mic = Ti2E()
utils.print_configs(mic.mmc)

### Experimental settings

`StimTreatment` names the stimulation channel, its group, and the
device/property that sets its power. On the Ti2-E these are the
`GFP` preset, the `Channel` group and the XT600 LED intensity, read
from `mic.POWER_PROPERTIES`.

In [ ]:
## Timing and frame counts
N_FRAMES_PHASE_1 = 100  # pre-drug
N_FRAMES_PHASE_2 = 150  # post-drug
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0
TIME_BETWEEN_TIMESTEPS = 60  # s; RTMSequence: time_plan={"interval": 60.0, ...}
TIME_PER_FOV = 3.75  # s; RTMSequence: utils.apply_fov_batching(events, 3.75)

## df_acquire bookkeeping
ADD_STIM_EXPOSURE_GROUP = False  # True: save per-FOV last-stim-exposure info
REGULAR_SPACING_BETWEEN_STIMULATIONS = False

## Storage
base_path = os.path.join(os.path.expanduser("~"), "faro_data")  # TODO: data folder
experiment_name = "2026-11-16_Priming_FGFR_U0126"
path = os.path.join(base_path, experiment_name)
WRITER = "omezarr"  # or "tiff"

## Imaging channels, on every frame (group: the Channel group of the scope)
channels = [
    Channel(config="iRFP", exposure=300),  # H2B-miRFP, segmented
    Channel(config="RFP", exposure=200),  # ERK-KTR-mScarlet3
]

## Optocheck, acquired as reference channel on optocheck_timepoints
channel_optocheck = Channel(config="YFP_Custom", exposure=600)
optocheck_timepoints = (N_FRAMES_PHASE_2 - 1,)  # timesteps of phase 2

## Conditions assigned to the FOVs
condition = ["Drug"]
n_fovs_per_well = None  # an int for well-plate experiments

## Stimulation: GFP preset, power through the XT600 LED intensity
LED_DEVICE, LED_INTENSITY = mic.POWER_PROPERTIES["GFP"]

stim_phase_1 = [
    StimTreatment(
        treatment_name="Priming Phase 1 pre Drug",
        stim_timestep=tuple(range(10, 100)),
        stim_exposure_list=(100,) * 90,  # one exposure per stim timestep
        stim_power=10,
        stim_channel_name="GFP",
        stim_channel_group=mic.CHANNEL_GROUP,
        stim_channel_device_name=LED_DEVICE,
        stim_channel_power_property_name=LED_INTENSITY,
    )
]

stim_phase_2 = [
    StimTreatment(
        treatment_name="Sustained Phase 2 post Drug",
        stim_timestep=tuple(range(30, 120)),
        stim_exposure_list=100,  # repeated with auto_repeat_stim_exposure
        auto_repeat_stim_exposure=True,
        stim_power=10,
        stim_channel_name="GFP",
        stim_channel_group=mic.CHANNEL_GROUP,
        stim_channel_device_name=LED_DEVICE,
        stim_channel_power_property_name=LED_INTENSITY,
    )
]

for stim_phase in [stim_phase_1, stim_phase_2]:
    utils.print_stim_exposures_timesteps(stim_phase)

### Pipeline

In [ ]:
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.ref import RefFE
from faro.segmentation.cellpose_v4 import CellposeV4
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

# Cellpose needs `uv sync --extra cellpose`. Without it, OtsuSegmentator works
# for a bright nuclear marker:
# from faro.segmentation.base import OtsuSegmentator;  CellposeV4() -> OtsuSegmentator()

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=CellposeV4(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=RefFE(used_mask="labels"),
)

### GUI: napari and Micro-Manager

Opens napari with the napari-micromanager widget. Use it for live view and
to pick the FOV positions in the MDA widget; the next steps read them from
there.

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_wdg)

### Build the acquisition DataFrames

- `generate_fov_positions(mic, viewer=viewer)` reads the FOV positions of the napari MDA widget.
- `generate_df_acquire(fovs, ...)` builds one row per FOV and timestep with channels, timing and metadata. `phase_id` goes into the file names, so the phases never share one.
- `apply_stim_treatments_to_df_acquire(df, stim, ...)` maps the `StimTreatment` objects onto the rows.

With RTMSequence all of this is one `RTMSequence(...)` per phase.

In [ ]:
fovs = utils.generate_fov_positions(mic, viewer=viewer)

# Phase 1 (pre-drug)
df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

In [ ]:
# Phase 2 (post-drug), with the optocheck channel on optocheck_timepoints
df_acquire_2 = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

### Run the experiment

Convert each DataFrame with `df_to_events()`. Phase 1 starts with
`run_experiment`; phase 2 uses `continue_experiment` on the same
controller, which keeps the tracking state and appends to the same
store. Pipette the drug between the two cells.

In [ ]:
from faro.core.controller import Controller
from faro.core.conversion import df_to_events
from faro.core.writers import OmeZarrWriter, TiffWriter

for _ in range(int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)
else:
    writer = TiffWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer)

events = df_to_events(df_acquire)
handle = ctrl.run_experiment(events, stim_mode="current")

In [ ]:
from faro.widgets import ExperimentStatusWidget

viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)

In [ ]:
# Pause point: wait for phase 1, pipette the drug, then run this cell.
handle.wait()
events_2 = df_to_events(df_acquire_2)
handle_2 = ctrl.continue_experiment(events_2, stim_mode="current")

In [ ]:
handle_2.wait()
ctrl.finish_experiment()
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks